# 性能分析与流批对比

本 notebook 用于分析流处理和批处理的性能数据，生成图表用于课程报告。

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

# 设置中文字体
matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

plt.style.use('seaborn-v0_8-whitegrid')

## 1. 流处理延迟分析

In [ ]:
# 加载流处理性能指标
perf_data = []
with open('../output/stream_results/perf_metrics.json', 'r') as f:
    for line in f:
        if line.strip():
            perf_data.append(json.loads(line))

perf_df = pd.DataFrame(perf_data)
print(f'共 {len(perf_df)} 个批次的性能数据')
perf_df.head()

In [ ]:
# 延迟趋势图
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(perf_df['batch_num'], perf_df['avg_latency_ms'], label='平均延迟', marker='o', markersize=3)
ax.plot(perf_df['batch_num'], perf_df['p95_latency_ms'], label='P95 延迟', marker='s', markersize=3)
ax.plot(perf_df['batch_num'], perf_df['p99_latency_ms'], label='P99 延迟', marker='^', markersize=3)
ax.set_xlabel('批次编号')
ax.set_ylabel('延迟 (ms)')
ax.set_title('Spark Structured Streaming 端到端延迟')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/screenshots/latency_trend.png', dpi=150)
plt.show()

In [ ]:
# 吞吐量趋势图
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(perf_df['batch_num'], perf_df['throughput'], color='green', marker='o', markersize=3)
ax.axhline(y=perf_df['throughput'].mean(), color='red', linestyle='--', label=f"平均: {perf_df['throughput'].mean():.0f} 条/秒")
ax.set_xlabel('批次编号')
ax.set_ylabel('吞吐量 (条/秒)')
ax.set_title('Spark Structured Streaming 吞吐量')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/screenshots/throughput_trend.png', dpi=150)
plt.show()

## 2. 批处理性能

In [ ]:
# 加载批处理性能报告
with open('../output/batch_results/perf_report.json', 'r') as f:
    batch_perf = json.load(f)

print('批处理性能报告:')
for k, v in batch_perf.items():
    print(f'  {k}: {v}')

In [ ]:
# 批处理各阶段耗时
stages = {
    '数据加载': batch_perf['load_time_sec'],
    '热门商品': batch_perf['hot_products_time_sec'],
    '窗口统计': batch_perf['windowed_products_time_sec'],
    '热搜词': batch_perf['hot_keywords_time_sec'],
    '分类统计': batch_perf['category_stats_time_sec'],
}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stages.keys(), stages.values(), color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6'])
ax.set_ylabel('耗时 (秒)')
ax.set_title('批处理各阶段耗时')
for bar, val in zip(bars, stages.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.2f}s', ha='center')
plt.tight_layout()
plt.savefig('../docs/screenshots/batch_stages.png', dpi=150)
plt.show()

## 3. 流处理 vs 批处理对比

In [ ]:
# 对比图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 延迟对比
stream_avg_latency = perf_df['avg_latency_ms'].mean() / 1000  # 转为秒
batch_latency = batch_perf['total_time_sec']

axes[0].bar(['流处理', '批处理'], [stream_avg_latency, batch_latency], 
            color=['#3498db', '#e74c3c'])
axes[0].set_ylabel('延迟 (秒)')
axes[0].set_title('首条结果延迟对比')
for i, v in enumerate([stream_avg_latency, batch_latency]):
    axes[0].text(i, v + 0.1, f'{v:.2f}s', ha='center')

# 吞吐量对比
stream_throughput = perf_df['throughput'].mean()
batch_throughput = batch_perf['throughput_per_sec']

axes[1].bar(['流处理', '批处理'], [stream_throughput, batch_throughput],
            color=['#3498db', '#e74c3c'])
axes[1].set_ylabel('吞吐量 (条/秒)')
axes[1].set_title('吞吐量对比')
for i, v in enumerate([stream_throughput, batch_throughput]):
    axes[1].text(i, v + 10, f'{v:.0f}', ha='center')

plt.tight_layout()
plt.savefig('../docs/screenshots/stream_vs_batch.png', dpi=150)
plt.show()

## 4. 统计汇总

In [ ]:
print('=' * 50)
print('流处理性能汇总')
print('=' * 50)
print(f"平均延迟:  {perf_df['avg_latency_ms'].mean():.0f} ms")
print(f"P95 延迟:  {perf_df['p95_latency_ms'].mean():.0f} ms")
print(f"P99 延迟:  {perf_df['p99_latency_ms'].mean():.0f} ms")
print(f"平均吞吐:  {perf_df['throughput'].mean():.0f} 条/秒")
print()
print('=' * 50)
print('批处理性能汇总')
print('=' * 50)
print(f"总耗时:    {batch_perf['total_time_sec']:.2f} s")
print(f"吞吐量:    {batch_perf['throughput_per_sec']:,} 条/秒")
print(f"总记录:    {batch_perf['total_records']:,} 条")